In [11]:
using LowLevelFEM, LinearAlgebra

In [12]:
structured_box_mesh(n=40)

mat = Material("body")

Material("body", :Hooke, 200000.0, 0.3, 115384.61538461536, 76923.07692307692, 166666.66666666663, 7.85e-9, 45.0, 4.2e8, 1.2e-5, 1.0e-7, 0.1, 1.0)

In [13]:
probE = Problem([mat], type=:Solid)

GC.gc()
@time K11 = stiffnessMatrix(probE)

  5.858445 seconds (35.26 M allocations: 14.546 GiB, 21.82% gc time)


sparse([1, 2, 3, 139, 140, 141, 142, 143, 144, 1078  …  206643, 206644, 206645, 206646, 206758, 206759, 206760, 206761, 206762, 206763], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  206763, 206763, 206763, 206763, 206763, 206763, 206763, 206763, 206763, 206763], [1175.213675213666, 400.6410256410224, -400.6410256410229, 267.0940170940177, 200.32051282051268, 80.12820512820247, 267.09401709401453, -80.12820512820421, -200.3205128205119, -534.1880341880329  …  -53.418803418802966, -5.400124791776761e-13, 1.2789769243681803e-12, 1068.3760683760836, 3.225864020350855e-12, 6.536993168992922e-13, 1068.376068376087, 4.433786671143025e-12, -5.115907697472721e-13, 9401.709401709406], 206763, 206763)

In [14]:
probH = Problem([mat], type=:HeatConduction)

GC.gc()
@time K12 = heatConductionMatrix(probH)

  6.534881 seconds (48.93 M allocations: 5.389 GiB, 23.99% gc time)


sparse([1, 47, 48, 360, 1959, 3557, 8082, 67401, 2, 9  …  9601, 9602, 67360, 67361, 67399, 67400, 68881, 68882, 68920, 68921], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2  …  68921, 68921, 68921, 68921, 68921, 68921, 68921, 68921, 68921, 68921], [0.3749999999999984, -1.7694179454963432e-16, -6.049848122469115e-17, 1.942890293094024e-16, -0.09374999999999999, -0.09374999999999914, -0.09374999999999958, -0.09374999999999963, 0.37500000000000006, 3.642919299551295e-17  …  -0.18749999999999956, -2.932116355269798e-15, -0.09375000000000086, -0.18750000000000044, -0.18750000000000078, 9.883587004377858e-15, -0.18750000000000133, 6.3267533573219126e-15, 6.476373257124912e-15, 3.0000000000000053], 68921, 68921)

In [15]:
Pu = Problem([mat], type=:VectorField, dim=3, field=:u)

μ = mat.μ
λ = mat.λ
Dμ = Diagonal([2μ, 2μ, 2μ, μ, μ, μ])

GC.gc()
@time K21 = ∫(SymGrad(Pu) ⋅ Dμ ⋅ SymGrad(Pu) + Div(Pu) ⋅ λ ⋅ Div(Pu))

  6.476720 seconds (8.19 M allocations: 19.041 GiB, 11.29% gc time)


sparse([1, 2, 3, 139, 140, 141, 142, 143, 144, 1078  …  206643, 206644, 206645, 206646, 206758, 206759, 206760, 206761, 206762, 206763], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  206763, 206763, 206763, 206763, 206763, 206763, 206763, 206763, 206763, 206763], [1175.2136752136662, 400.64102564102234, -400.6410256410228, 267.09401709401766, 200.32051282051265, 80.12820512820241, 267.0940170940145, -80.12820512820414, -200.32051282051196, -534.1880341880328  …  -53.41880341880301, -4.405364961712621e-13, 1.2221335055073723e-12, 1068.3760683760836, 3.069544618483633e-12, 6.821210263296962e-13, 1068.3760683760866, 4.575895218295045e-12, -7.105427357601002e-13, 9401.709401709406], 206763, 206763)

In [16]:
D = [λ+2μ λ λ 0 0 0; λ λ+2μ λ 0 0 0; λ λ λ+2μ 0 0 0; 0 0 0 μ 0 0; 0 0 0 0 μ 0; 0 0 0 0 0 μ]

GC.gc()
@time K21 = ∫(SymGrad(Pu) ⋅ D ⋅ SymGrad(Pu))

  3.679722 seconds (4.61 M allocations: 9.374 GiB)


sparse([1, 2, 3, 139, 140, 141, 142, 143, 144, 1078  …  206643, 206644, 206645, 206646, 206758, 206759, 206760, 206761, 206762, 206763], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1  …  206763, 206763, 206763, 206763, 206763, 206763, 206763, 206763, 206763, 206763], [1175.213675213666, 400.64102564102234, -400.64102564102285, 267.0940170940176, 200.32051282051265, 80.12820512820242, 267.0940170940145, -80.12820512820416, -200.3205128205119, -534.1880341880327  …  -53.41880341880304, -4.263256414560601e-13, 1.2931877790833823e-12, 1068.3760683760836, 3.140598892059643e-12, 6.536993168992922e-13, 1068.3760683760866, 4.604316927725449e-12, -7.389644451905042e-13, 9401.709401709404], 206763, 206763)

In [17]:
PT = Problem([mat], type=:ScalarField, dim=3, field=:T)

k = mat.k

GC.gc()
@time K22 = ∫(Grad(PT) ⋅ k ⋅ Grad(PT))

  1.294818 seconds (3.58 M allocations: 1.164 GiB)


sparse([1, 47, 48, 360, 1959, 3557, 8082, 67401, 2, 9  …  9601, 9602, 67360, 67361, 67399, 67400, 68881, 68882, 68920, 68921], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2  …  68921, 68921, 68921, 68921, 68921, 68921, 68921, 68921, 68921, 68921], [0.3749999999999984, -1.7867651802561113e-16, -6.223320470066795e-17, 1.8908485888147197e-16, -0.09374999999999996, -0.09374999999999913, -0.0937499999999996, -0.09374999999999964, 0.37500000000000006, 3.642919299551295e-17  …  -0.18749999999999956, -2.9303816317938214e-15, -0.09375000000000086, -0.18750000000000044, -0.18750000000000078, 9.887056451329812e-15, -0.18750000000000133, 6.325018633845936e-15, 6.4707354058279876e-15, 3.0000000000000053], 68921, 68921)

In [18]:
norm(K11.A - K21.A) / norm(K11.A)

1.985212982295291e-16

In [19]:
norm(K12.A - K22.A) / norm(K12.A)

8.035731222110114e-17

In [24]:
@time f = ∫(Pu ⋅ [1, 1, 1])

  1.681790 seconds (4.61 M allocations: 1.015 GiB, 12.95% gc time)


nodal VectorField
[1.953124999999996e-6; 1.953124999999996e-6; … ; 1.5625000000000055e-5; 1.5625000000000055e-5;;]